In [ ]:
# @title
#Implementing all the necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error



In [ ]:
# @title
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)



In [ ]:
# @title
# Task 1:
df = pd.read_csv(f"{path}/Q1_data.csv")

In [ ]:
# @title
# Task 2:
df.head()

In [ ]:
# Task 3:
df.info()

In [ ]:
# Task 4:
df.describe()

In [ ]:
# Task 5:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Task 1:
df.drop(columns=['Order_ID'], inplace=True)

In [ ]:
# Task 2:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
print(missing_data)

# Dropping rows with missing values if they are few
df.dropna(inplace=True)

In [ ]:
# Task 3:
df.drop_duplicates(inplace=True)

In [ ]:
# Task 4:
from sklearn.preprocessing import StandardScaler, LabelEncoder
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

df

In [ ]:
# Task 5:
scaler = StandardScaler()
df[df.columns] = scaler.fit_transform(df)

In [ ]:
# Task 6: Write your code here:
import seaborn as sns
# Checking for target imbalance
delivery_time_counts = df['Delivery_Time'].value_counts()
plt.figure(figsize=(10, 5))
sns.histplot(delivery_time_counts, bins=30)
plt.title('Target Variable Distribution')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1:
X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mae_scores.append(mae)

average_mae = np.mean(mae_scores)
print(f'Average MAE across folds: {average_mae}')


In [ ]:
# Task 1:
#Average feature importance across folds
importances = model.feature_importances_
features = X.columns
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
plt.title('Feature Importances')
plt.bar(range(X.shape[1]), importances[indices], align='center')
plt.xticks(range(X.shape[1]), features[indices], rotation=90)
plt.xlim([-1, X.shape[1]])
plt.show()

In [ ]:
# Task 2:
predictions = model.predict(X)
plt.figure(figsize=(10, 5))
plt.hist(predictions, bins=50, edgecolor='black')
plt.title('Predicted Delivery Time Distribution')
plt.xlabel('Predicted Time (minutes)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
%pip install catboost

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    rf_model = RandomForestRegressor(random_state=42)
    cb_model = CatBoostRegressor(verbose=0)

    rf_model.fit(X_train, y_train) #Training RandomForest Model
    cb_model.fit(X_train, y_train) #Training CatBoost Model

    rf_predictions = rf_model.predict(X_test)
    cb_predictions = cb_model.predict(X_test)

    # Average predictions
    avg_predictions = (rf_predictions + cb_predictions) / 2

    mae = mean_absolute_error(y_test, avg_predictions)
    mae_scores.append(mae)

average_mae = np.mean(mae_scores)
print(f'Average MAE across folds with ensemble: {average_mae}')
